# NTGEN: General Nanotube Generation from the Alexandria 1D Database

**Google Colab notebook — requires a GPU runtime.**
Before running: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

This notebook generates new general (multi-element) nanotube candidates with the
NTGEN diffusion model, seeded by **real 1D nanotube structures** from the
Alexandria database. It uses SCIGEN-style mask-constrained denoising
("Pathway 3") — the pretrained mp_20 diffusion model is used as-is, **no
retraining required**.

**Pipeline**: sample a real nanotube template from
`data/alx_1D/nanotube_templates.npz` (2,216 structures distilled from the 7,002
ASE structures in `alexandria_1d_nanotubes.pkl`) → `SC_DBTemplate` pins the
template's atoms (per-atom species + fractional coords + cell) as the known
skeleton via `mask_x`/`mask_l`/`mask_t` → `sample_scigen` reverse diffusion
re-imposes the pinned atoms at every step while the model denoises the remaining
decorating atoms → `pymatgen` → CIF.

> **Opening this notebook in Colab**: once the repo is public, open
> [colab.research.google.com/github/3venthatguy/NTU-IQM/blob/main/models/NTGEN_generation/ntgen_generation.ipynb](https://colab.research.google.com/github/3venthatguy/NTU-IQM/blob/main/models/NTGEN_generation/ntgen_generation.ipynb)
> — or in Colab use **File → Open notebook → GitHub** and paste the repo URL.


In [ ]:
# takes long (~5-10 min first run) — clones repo, installs deps, downloads model

# --- Run once per Colab session ---
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/3venthatguy/NTU-IQM.git'
REPO_DIR = '/content/NTU-IQM'
PROJECT_DIR = f'{REPO_DIR}/models/NTGEN-edit'

# Clone repo if needed (the repo must be public for an anonymous clone)
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])

# The package dir is named ntgent/ but all imports & hydra targets say scigen.*
# -> self-heal with a symlink.
if not os.path.exists(f'{PROJECT_DIR}/scigen') and os.path.exists(f'{PROJECT_DIR}/ntgent'):
    os.symlink('ntgent', f'{PROJECT_DIR}/scigen')
    print('created symlink scigen -> ntgent')

# Install PyG extensions (need matching torch+CUDA wheels)
try:
    import torch_scatter
except ImportError:
    import torch
    _vp = torch.__version__.split('+')[0].split('.')
    tv = f'{_vp[0]}.{_vp[1]}.0'
    if torch.version.cuda:
        cv = 'cu' + torch.version.cuda.replace('.', '')
    else:
        cv = 'cpu'
    whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
    print(f'Installing PyG extensions from: {whl}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch-scatter', 'torch-sparse', 'torch-cluster',
                           '-f', whl])

# Remaining dependencies (torch itself comes preinstalled on Colab)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'hydra-core', 'omegaconf', 'pytorch-lightning',
                       'pymatgen', 'torch-geometric', 'einops',
                       'p_tqdm', 'pyxtal', 'pathos', 'python-dotenv',
                       'scipy', 'scikit-learn', 'networkx'])

# Download the pretrained SCIGEN mp_20 checkpoint from Figshare if not present
from pathlib import Path
MODEL_PATH = Path(PROJECT_DIR) / 'models' / 'mp_20'
if not MODEL_PATH.exists() or not list(MODEL_PATH.glob('*.ckpt')):
    import json, zipfile
    from urllib.request import urlopen, urlretrieve
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    article = json.loads(urlopen('https://api.figshare.com/v2/articles/27778134').read().decode())
    for f in article.get('files', []):
        dest = MODEL_PATH / f['name']
        print(f'Downloading {f["name"]}...')
        urlretrieve(f['download_url'], str(dest))
        if dest.suffix == '.zip':
            with zipfile.ZipFile(dest, 'r') as zf:
                zf.extractall(MODEL_PATH)
            dest.unlink()

os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')


---
## 1. How NTGEN works

NTGEN is the SCIGEN diffusion framework focused purely on **nanotube
generation**. The diffusion model generates a complete crystal (lattice + atom
positions + species) from noise; at every denoising step the constraint masks
**re-impose the known nanotube skeleton**, so the model only has to place the
remaining "decorating" atoms around it.

### Available structural constraints (`sc_dict` in `script/sc_utils.py`)

| Code | Constraint | Known atoms | Description |
|------|------------|-------------|-------------|
| `alx` | **Alexandria template** | whole template | A real 1D nanotube from the Alexandria database, pinned atom-by-atom (this notebook) |
| `ntb` | Parametric nanotube | ring skeleton | Synthetic skeleton tube, single element |
| `cnt` | Carbon nanotube | full wall | Rolled-graphene wall from chiral indices (n,m) |
| `van` | Vanilla | 1 | Unconstrained generation |

> **Dataset note**: run `'alx'` with a *general* dataset (`mp_20`). `carbon_24`
> would force every atom to carbon and flatten the template's real
> multi-element species.


---
## 2. Environment setup

Safe to re-run at any time (e.g. after a runtime restart — the clone and
installs from the setup cell above persist for the session).


In [ ]:
import os, sys

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/NTU-IQM/models/NTGEN-edit')
NOTEBOOK_DIR = '/content/NTU-IQM/models/NTGEN_generation'

assert os.path.exists(PROJECT_DIR), (
    f'{PROJECT_DIR} not found — run the setup cell at the top first.')

# scigen -> ntgent symlink (self-heal if the setup cell was skipped)
if not os.path.exists(f'{PROJECT_DIR}/scigen') and os.path.exists(f'{PROJECT_DIR}/ntgent'):
    os.symlink('ntgent', f'{PROJECT_DIR}/scigen')

os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
os.environ.setdefault('HYDRA_JOBS', PROJECT_DIR)
os.environ.setdefault('WANDB_DIR', os.path.join(PROJECT_DIR, 'wandb'))
os.environ.setdefault('WANDB_MODE', 'disabled')   # no wandb login prompts
os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)

for p in (PROJECT_DIR, os.path.join(PROJECT_DIR, 'script')):
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(PROJECT_DIR)   # gen_utils opens ./data/... relative to cwd

# The Alexandria template cache ships with the repo (648 KB, built once from
# the raw 48 MB ASE pickle by data/alx_1D/build_templates.py).
cache = os.path.join(PROJECT_DIR, 'data', 'alx_1D', 'nanotube_templates.npz')
assert os.path.exists(cache), (
    f'{cache} missing — the clone is incomplete or the cache was never '
    'committed. Re-run the setup cell, or rebuild with '
    'data/alx_1D/build_templates.py (requires the raw pickle).')

print(f'Working directory: {os.getcwd()}')
print(f'Template cache:    {cache} ({os.path.getsize(cache)/1024:.0f} KB)')


---
## 3. Load the pretrained model

We load the SCIGEN mp_20 diffusion model using Hydra for configuration and
manual state-dict loading for compatibility with any PyTorch Lightning version,
then attach the constraint-aware sampler `sample_scigen`.


In [ ]:
import torch
import numpy as np
import hydra
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from pathlib import Path

import scigen
print(f'scigen imported from: {scigen.__path__[0]}')


def load_model_for_inference(model_path, device='cpu'):
    """Load the SCIGEN pretrained model for inference.

    Uses manual state_dict loading instead of pytorch_lightning's
    load_from_checkpoint, so it works with any PL version.
    """
    model_path = Path(model_path)

    # Clear previous hydra state (allows re-running this cell)
    GlobalHydra.instance().clear()

    # Load model config from hparams.yaml
    with initialize_config_dir(config_dir=str(model_path.resolve()), version_base=None):
        cfg = compose(config_name='hparams')

    # Instantiate model architecture (empty weights)
    model = hydra.utils.instantiate(
        cfg.model, optim=cfg.optim, data=cfg.data,
        logging=cfg.logging, _recursive_=False,
    )

    # Find checkpoint file
    ckpts = sorted(model_path.glob('*.ckpt'))
    if not ckpts:
        raise FileNotFoundError(f'No .ckpt files found in {model_path}')

    ckpt_path = next((c for c in ckpts if 'last' in c.name), ckpts[-1])
    print(f'Loading checkpoint: {ckpt_path.name}')

    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['state_dict'], strict=False)

    # Load data scalers
    for attr, fname in [('lattice_scaler', 'lattice_scaler.pt'),
                        ('scaler', 'prop_scaler.pt')]:
        fpath = model_path / fname
        if fpath.exists():
            setattr(model, attr, torch.load(fpath, map_location='cpu', weights_only=False))

    model = model.to(device)
    model.eval()
    return model, cfg


In [ ]:
MODEL_PATH = Path(PROJECT_DIR) / 'models' / 'mp_20'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type != 'cuda':
    print('WARNING: no GPU detected — generation will be very slow. '
          'Use Runtime -> Change runtime type -> T4 GPU.')

model, cfg = load_model_for_inference(MODEL_PATH, device=device)

# Attach the constraint-aware sampling method
from scigen.pl_modules.diffusion_w_type import sample_scigen
model.sample_scigen = sample_scigen.__get__(model)

num_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {num_params:,} parameters')


---
## 4. Generation settings

- `SC_TYPE='alx'`: each candidate starts from one **real Alexandria 1D
  nanotube** drawn uniformly among templates whose atom count fits
  `natm_range` (with at least one slot left for a model-placed decorating
  atom). The template's per-atom species are pinned exactly — multi-element
  skeletons stay multi-element. If no template fits the range, that draw
  silently falls back to the parametric `SC_Nanotube` (`'ntb'`).
- `DATASET='mp_20'` supplies the atom-count distribution (up to 20 atoms) and
  keeps atom types free for the diffusion model — required for multi-element
  templates.
- `KNOWN_SPECIES` only feeds the bond-length sampler; `SC_DBTemplate` ignores
  it (the template carries its own species/geometry).
- `reduced_mask=False` is required: template atoms are pinned per-coordinate
  (`mask_x` has shape `(N, 3)`).


In [ ]:
# ============================================================
#  EDIT THESE PARAMETERS to try different generation settings
# ============================================================
SC_TYPE       = 'alx'    # 'alx' Alexandria template | 'ntb' parametric | 'cnt' | 'van'
DATASET       = 'mp_20'  # general dataset: preserves multi-element template species
KNOWN_SPECIES = ['Fe']   # bond-length sampler only; ignored by 'alx' templates
BATCH_SIZE    = 4        # structures per batch (keep small for Colab T4)
NUM_BATCHES   = 1        # number of batches
# ============================================================

FRAC_Z  = 0.5            # unused by 'alx' (template sets coords); kept for 'ntb'
STEP_LR = 5e-6           # diffusion step size
SEED    = 42

# Atom count ranges per structural constraint
SC_NATM_RANGE = {'alx': [1, 20], 'ntb': [4, 20], 'cnt': [1, 20], 'van': [1, 20]}

natm_range = SC_NATM_RANGE.get(SC_TYPE, [1, 20])
total_structures = BATCH_SIZE * NUM_BATCHES

print('Generation settings:')
print(f'  Constraint:     {SC_TYPE}')
print(f'  Dataset:        {DATASET}')
print(f'  Structures:     {total_structures}')
print(f'  Atoms per cell: {natm_range[0]}-{natm_range[1]}')


---
## 5. Generate

`SampleDataset` draws one Alexandria template per candidate and builds the
constraint masks; the loop below runs reverse diffusion with the template
re-imposed at every step. The printout before generation shows exactly which
skeleton was pinned for each candidate.


In [ ]:
# takes long (~20-60 sec on a T4) — runs diffusion sampling

import time
from tqdm.auto import tqdm
from torch_geometric.data import DataLoader
from gen_utils import SampleDataset
from sc_utils import chemical_symbols

# Build the sample dataset with the nanotube-template constraint
test_set = SampleDataset(
    dataset=DATASET,
    natm_range=natm_range,
    total_num=total_structures,
    bond_sigma_per_mu=None,
    use_min_bond_len=False,
    known_species=KNOWN_SPECIES,
    sc_list=[SC_TYPE],
    frac_z=FRAC_Z,
    c_vec_cons={'scale': None, 'vert': False},
    reduced_mask=False,
    seed=SEED,
    device=device,
)

# Peek at the drawn templates: pinned skeleton size + species per candidate
print('Pinned skeletons:')
for i in range(len(test_set)):
    d = test_set[i]
    K = int(d.num_known)
    species = sorted({chemical_symbols[int(z)] for z in d.atom_types_known[:K]})
    print(f'  [{i}] {K} pinned atoms {species} -> {int(d.num_atoms[0])} total')

test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

# Run diffusion sampling
all_frac_coords, all_atom_types, all_lattices = [], [], []
all_num_atoms, all_num_known = [], []

print(f'\nRunning diffusion (step_lr={STEP_LR})...')
start_time = time.time()

for idx, batch in enumerate(tqdm(test_loader, desc='Generating structures')):
    if torch.cuda.is_available():
        batch.cuda()
    outputs, traj = model.sample_scigen(batch, step_lr=STEP_LR)

    all_frac_coords.append(outputs['frac_coords'].detach().cpu())
    raw_types = outputs['atom_types'].detach().cpu()
    if raw_types.dim() == 2:
        raw_types = raw_types.argmax(dim=-1) + 1
    all_atom_types.append(raw_types)
    all_lattices.append(outputs['lattices'].detach().cpu())
    all_num_atoms.append(outputs['num_atoms'].detach().cpu())
    all_num_known.append(outputs['num_known'].detach().cpu())

    print(f'  Batch {idx + 1}/{len(test_loader)} complete')

elapsed = time.time() - start_time
print(f'\nGeneration complete in {elapsed:.1f}s ({elapsed / total_structures:.1f}s per structure)')

frac_coords = torch.cat(all_frac_coords, dim=0)
atom_types = torch.cat(all_atom_types, dim=0)
lattices = torch.cat(all_lattices, dim=0)
num_atoms = torch.cat(all_num_atoms, dim=0)
num_known = torch.cat(all_num_known, dim=0)
print('Atoms per structure:', num_atoms.tolist(), '| pinned skeleton:', num_known.tolist())


---
## 6. Inspect the results

Convert the raw diffusion outputs (fractional coords, atom types, lattices) to
`pymatgen` `Structure` objects, then take a quick 3D look at one candidate.


In [ ]:
from pymatgen.core.lattice import Lattice
from pymatgen.core.structure import Structure


def lattices_to_params(lat):
    """(3,3) lattice matrix -> (lengths[3], angles_deg[3])."""
    lengths = np.linalg.norm(lat, axis=1)
    angles = np.zeros(3)
    for i in range(3):
        j, k = (i + 1) % 3, (i + 2) % 3
        cos = np.dot(lat[j], lat[k]) / (lengths[j] * lengths[k])
        angles[i] = np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))
    return lengths, angles


structures = []
start = 0
for i in range(num_atoms.shape[0]):
    n_i = int(num_atoms[i])
    coords_i = frac_coords[start:start + n_i].numpy()
    types_i = atom_types[start:start + n_i].numpy()
    start += n_i
    lengths_i, angles_i = lattices_to_params(lattices[i].numpy())
    species = [chemical_symbols[int(t)] for t in types_i]
    try:
        structure = Structure(
            Lattice.from_parameters(*lengths_i, *angles_i),
            species, coords_i, coords_are_cartesian=False)
        structures.append(structure)
        print(f'  [{i}] {structure.composition.reduced_formula}: '
              f'{n_i} atoms, a={lengths_i[0]:.2f} b={lengths_i[1]:.2f} c={lengths_i[2]:.2f} A')
    except Exception as e:
        structures.append(None)
        print(f'  [{i}] conversion failed ({e})')

print(f'\n{sum(s is not None for s in structures)} / {len(structures)} structures converted')


In [ ]:
import matplotlib.pyplot as plt

s0 = next((s for s in structures if s is not None), None)
if s0 is not None:
    xyz = s0.cart_coords
    zs = [site.specie.Z for site in s0]
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(projection='3d')
    ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=zs, cmap='viridis', s=60)
    ax.set_title(s0.composition.reduced_formula)
    plt.show()


---
## 6b. Diffusion trajectory (how a candidate forms)

`sample_scigen` returns a per-step trajectory as its second value (`traj`). The
cells below visualise structure 0 evolving from noise (t=1.0) to the final
crystal (t=0.0). The **pinned nanotube skeleton stays fixed at every step** (it is
re-imposed by the constraint masks), while the model places the decorating atoms
around it — this is the visual signature of the Pathway-3 inpainting.

Red-ringed atoms are the constrained skeleton sites.


In [ ]:
# --- Diffusion trajectory filmstrip (matplotlib) -------------------------------
# NTGEN has no PyVista `crystal_viz`, so we render frames with matplotlib.
import numpy as np
import matplotlib.pyplot as plt
from sc_utils import chemical_symbols

assert 'traj' in globals(), (
    "`traj` not found — run the generation cell (Section 5) first. "
    "It is the 2nd return value of model.sample_scigen and holds the trajectory "
    "of the LAST batch generated.")

traj_coords    = traj['all_frac_coords'].detach().cpu()   # (T+1, total_atoms, 3)
traj_lattices  = traj['all_lattices'].detach().cpu()      # (T+1, n_struct, 3, 3)
traj_types     = traj['atom_types'].detach().cpu()        # (T+1, total_atoms) 1-indexed Z
traj_num_atoms = traj['num_atoms'].detach().cpu()
traj_num_known = traj['num_known'].detach().cpu().flatten()

n_steps  = traj_coords.shape[0]
na_first = int(traj_num_atoms[0].item())                  # atoms in structure 0
nk_first = int(traj_num_known[0].item()) if traj_num_known.numel() > 0 else 0

n_frames = min(8, n_steps)
frame_indices = np.linspace(0, n_steps - 1, n_frames, dtype=int)
print(f'Trajectory: {n_steps} steps, showing {n_frames} frames')
print(f'Structure 0: {na_first} atoms ({nk_first} pinned skeleton)')


def frame_to_cart(step_idx):
    """Cartesian coords + per-atom Z for structure 0 at a trajectory step."""
    frac = traj_coords[step_idx, :na_first].numpy() % 1.0
    lat  = traj_lattices[step_idx, 0].numpy()             # 3x3 lattice matrix
    zs   = traj_types[step_idx, :na_first].int().tolist()
    return frac @ lat, zs                                 # (na,3), [Z...]


fig = plt.figure(figsize=(4 * min(4, n_frames), 4 * ((n_frames + 3) // 4)))
for fi, step_idx in enumerate(frame_indices):
    cart, zs = frame_to_cart(step_idx)
    ax = fig.add_subplot((n_frames + 3) // 4, min(4, n_frames), fi + 1, projection='3d')
    ax.scatter(cart[:, 0], cart[:, 1], cart[:, 2], s=90, c=zs, cmap='viridis',
               alpha=0.85, edgecolors='k', linewidths=0.4)
    if nk_first > 0:   # ring the pinned skeleton in red
        ax.scatter(cart[:nk_first, 0], cart[:nk_first, 1], cart[:nk_first, 2],
                   s=230, facecolors='none', edgecolors='red', linewidths=1.8)
    t_frac = step_idx / max(n_steps - 1, 1)
    ax.set_title(f'step {step_idx}  (t={1 - t_frac:.2f})', fontsize=9)
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])

plt.suptitle(f'Diffusion trajectory ({SC_TYPE}): noise -> nanotube candidate', y=1.0)
plt.tight_layout()
plt.show()


In [ ]:
# --- Animated GIF of the diffusion trajectory (matplotlib -> PIL) --------------
import io
from PIL import Image
from IPython.display import Image as IPyImage, display

n_anim = min(20, n_steps)
anim_indices = np.linspace(0, n_steps - 1, n_anim, dtype=int)

# Fix the view box to the final structure so the tube does not jump between frames
final_cart, _ = frame_to_cart(n_steps - 1)
pad = 1.0
lims = [(final_cart[:, d].min() - pad, final_cart[:, d].max() + pad) for d in range(3)]

print(f'Rendering {n_anim}-frame animation...')
anim_images = []
for step_idx in anim_indices:
    cart, zs = frame_to_cart(step_idx)
    fig = plt.figure(figsize=(4.5, 4.2))
    ax = fig.add_subplot(projection='3d')
    ax.scatter(cart[:, 0], cart[:, 1], cart[:, 2], s=90, c=zs, cmap='viridis',
               alpha=0.85, edgecolors='k', linewidths=0.4)
    if nk_first > 0:
        ax.scatter(cart[:nk_first, 0], cart[:nk_first, 1], cart[:nk_first, 2],
                   s=230, facecolors='none', edgecolors='red', linewidths=1.8)
    ax.set_xlim(*lims[0]); ax.set_ylim(*lims[1]); ax.set_zlim(*lims[2])
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    t_frac = step_idx / max(n_steps - 1, 1)
    ax.set_title(f't = {1 - t_frac:.2f}', fontsize=11)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=80, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    anim_images.append(Image.open(buf).convert('RGB'))

if len(anim_images) > 2:
    w, h = anim_images[0].size
    anim_images = [im.resize((w, h)) for im in anim_images]   # uniform frame size
    gif_buf = io.BytesIO()
    anim_images[0].save(gif_buf, format='GIF', save_all=True,
                        append_images=anim_images[1:], duration=300, loop=0)
    gif_buf.seek(0)
    display(IPyImage(data=gif_buf.read(), format='gif'))
    print(f'Animation: {len(anim_images)} frames, t = 1.0 (noise) -> 0.0 (crystal)')
else:
    print('Too few trajectory frames for an animation.')


---
## 7. Export

Save the candidates as CIF files, then download them as a zip archive
(Colab's disk is wiped when the runtime ends).


In [ ]:
output_dir = os.path.join(NOTEBOOK_DIR, 'generated_cifs')
os.makedirs(output_dir, exist_ok=True)

n_written = 0
for i, structure in enumerate(structures):
    if structure is None:
        continue
    formula = structure.composition.reduced_formula
    cif_path = os.path.join(output_dir, f'{SC_TYPE}_{formula}_{i:03d}.cif')
    structure.to(filename=cif_path, fmt='cif')
    print(f'Saved: {os.path.basename(cif_path)}')
    n_written += 1

print(f'\n{n_written} CIF files saved to {output_dir}')


In [ ]:
# Download CIF files as a zip archive (Colab only)
try:
    import shutil
    from google.colab import files
    zip_name = f'ntgen_{SC_TYPE}_cifs'
    zip_path = shutil.make_archive(os.path.join('/content', zip_name), 'zip', output_dir)
    files.download(zip_path)
    print(f'Downloading {zip_name}.zip')
except ImportError:
    print('Not running in Colab — skipping download.')
except Exception as e:
    print(f'Download failed: {e}')
    print(f'CIF files are available at: {output_dir}')


---
## 8. Try it yourself

Go back to **Section 4** and change the parameters, then re-run from Section 5:

- **More structures:** increase `BATCH_SIZE` or `NUM_BATCHES`
- **Other constraints:** set `SC_TYPE='ntb'` for parametric skeleton tubes
  (single element from `KNOWN_SPECIES`), or `'cnt'` for carbon nanotubes
  (best paired with `DATASET='carbon_24'`; see `05_ctgen_generation.ipynb`)
- **Different step size:** try `STEP_LR = 1e-5` and compare structure quality

### Notes

- **Template pool**: the cache keeps Alexandria structures with ≤ 64 atoms
  (2,216 of 7,002); with `natm_range=[1, 20]` about 370 templates are
  drawable. Raise `MAX_NATM` in `data/alx_1D/build_templates.py` and rebuild
  to widen the pool (the mp_20 checkpoint was trained on ≤ 20 atoms, so larger
  cells are out-of-distribution).
- **Fallback**: draws with no fitting template silently use the parametric
  `SC_Nanotube` — check the "Pinned skeletons" printout in Section 5 to see
  what was actually pinned.

### References

- **SCIGEN:** Okabe et al., "Structural constraint integration in a generative
  model for the discovery of quantum materials," *Nature Materials* (2025).
  [DOI](https://doi.org/10.1038/s41563-025-02355-y) |
  [GitHub](https://github.com/RyotaroOKabe/SCIGEN)
- **Alexandria database:** Schmidt et al., "Machine-learning-assisted
  determination of the global zero-temperature phase diagram of materials,"
  and the Alexandria 1D dataset. [alexandria.icams.rub.de](https://alexandria.icams.rub.de/)
- **pymatgen:** Ong et al., "Python Materials Genomics (pymatgen)," *Comput.
  Mater. Sci.* 68, 314–319 (2013).
  [GitHub](https://github.com/materialsproject/pymatgen)
